# Computer Vision - Part 11

### Computer Vision: AI Line Following

AI Line Following is a computer vision technique used in robotics and autonomous vehicles to enable them to follow a specific path or line. This technology relies on cameras and image processing algorithms to detect and track the line, allowing the robot or vehicle to navigate along it effectively.

Examples are car robots that can follow a painted line on the ground, or drones that can follow a specific path marked by a line. The AI algorithms analyze the video feed from the camera to identify the line's position and adjust the robot's movement accordingly, ensuring it stays on course. This technology is crucial for applications such as warehouse automation, delivery robots, and autonomous vehicles.

In [2]:
import cv2
import numpy as np

#### 1 – Offline Image Processing

Read a video/image sequence of a track, compute error `e(t)` over time, and plot it to visualize track difficulty.

- ROI: Bottom 40% of frame — focuses on the line ahead

- Otsu Threshold: Auto-selects threshold — robust under changing ligh

- Centroid: Finds the "center" of the line in the ROI

#### Error Convention

- e < 0 → line shifted left
- e > 0 → line shifted right
- e = 0 → line centered

In [3]:
def line_error_from_frame(frame, h, w):

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    roi = gray[int(h*0.6):h, :]
    blurred = cv2.GaussianBlur(roi, (5,5), 0)
    _, bw = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

    M = cv2.moments(bw)
    cx = int(M['m10'] / M['m00'])

    e = (cx - w/2.0) / (w/2.0) # ∈ [-1, 1]
    
    return e


#### 2 – Unicycle Simulator + P Controller

Simulate a unicycle robot tracking a sine-wave reference using a P controller on lateral error.

- `unicycle_step()`: Updates `(x, y, θ )` using velocity vand angular rate `ω`

- `lateral_error()`: Computes the signed perpendicular distance from robot to reference point

- `lookahead()`: Determines the reference point 0.3 m ahead on the reference — avoids control lag

In [4]:
def reference_line(t):
    x_ref = t
    y_ref = 0.5 * np.sin(0.5 * t)
    return x_ref, y_ref

def unicycle_step(x, y, th, v, w, dt):
    x += v * np.cos(th) * dt
    y += v * np.sin(th) * dt
    th += w * dt
    return x, y, th

def lateral_error(x, y, th, x_ref, y_ref):
    dx = x_ref - x
    dy = y_ref - y
    e = -np.sin(th) * dx + np.cos(th) * dy
    return e

Kp = 1.5 # P gain
v_const = 0.5 # forward speed (m/s)
dt = 0.05 # timestep (s)
t = 20.0 # total time (s)
x, y, th = 0.0, -0.5, 0.0 # initial state

# Per step:
x_ref, y_ref = reference_line(t + 0.3)
e = lateral_error(x, y, th, x_ref, y_ref)
w = Kp * e
x, y, th = unicycle_step(x, y, th, v_const, w, dt)

print(f"t={t:.2f}s, x={x:.2f}, y={y:.2f}, th={th:.2f} rad, e={e:.3f} m, w={w:.3f} rad/s")

t=20.00s, x=0.03, y=-0.50, th=0.01 rad, e=0.168 m, w=0.253 rad/s


#### 3 – Ackermann (Jet Racer) Simulator

Replace unicycle kinematics with Ackermann steering to simulate the Jet Racer platform.

- `ackermann_step()`: Updates pose using steering angle δand wheelbase `L = 0.25 m`

- `delta_max`: Mechanical limit: ±25° — steering is clipped to this range

- Convergence: Ackermann can't spin in place — large errors take longer to correct

In [5]:
def ackermann_step(x, y, th, v, delta, L, dt):
    x += v * np.cos(th) * dt
    y += v * np.sin(th) * dt
    th += (v / L) * np.tan(delta) * dt
    return x, y, th

# Steering command:
delta_max = np.radians(25) # max steering angle
delta = np.clip(Kp * e, -delta_max, delta_max)

print(f"t={t:.2f}s, x={x:.2f}, y={y:.2f}, th={th:.2f} rad, e={e:.3f} m, delta={np.degrees(delta):.1f} deg")

t=20.00s, x=0.03, y=-0.50, th=0.01 rad, e=0.168 m, delta=14.5 deg


#### 4 – Vision Pipeline + Real-Time Controller

Connect the OpenCV image pipeline to the unicycle controller in a live webcam loop — validate before deploying on hardware.

<img src="cpv-images/unicycle.png" width=800px height=350px />

The loop runs at ~30 fps. If the line is lost, error defaults to 0 (robot drives straight). Overlay shows e and ω values live on the frame.

In [13]:
def line_error_from_frame(frame, h, w):

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    roi = gray[int(h*0.6):h, :]
    blurred = cv2.GaussianBlur(roi, (5,5), 0)
    _, bw = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

    M = cv2.moments(bw)
    if M['m00'] == 0:
        return None, bw # line lost
    cx = int(M['m10'] / M['m00'])

    e = (cx - w/2.0) / (w/2.0) # ∈ [-1, 1]
    
    return e, bw

In [ ]:
# Frame as webcam
frame = cv2.imread('mammoth.jpg') # placeholder for webcam frame

h, w, _ = frame.shape
rx, ry, rth = 0.0, -0.5, 0.0 # robot state
ww = w # image width in pixels corresponds to 1 meter in world units

e, bw = line_error_from_frame(frame, h, w)
if e is None: 
    e = 0.0 # keep straight
    
w = Kp * e
rx, ry, rth = unicycle_step(rx, ry, rth, v_const, w, dt)

# Draw centroid dot (red):
if e is not None:
    cx = int((e * (w/2.0)) + (w/2.0))
    cy = int(h*0.6 + h*0.4/2)
    cv2.circle(frame, (cx, cy), 5, (0, 0, 255), -1)
    print(f"e={e:.3f}, w={w:.3f} rad/s")
else:
    print("Line lost, driving straight (e=0)")

e=0.097, w=0.145 rad/s


#### 5 – Jet Racer Road Following (NVIDIA/Waveshare)

Uses the jetracer SDK with a ResNet-18 regression model trained to predict a target point (x, y )on the road image.

1. Collect Data: Click target points in `road_following_data_collection.ipynb`

2. Train Model: ResNet-18 regression, ~10–20 min on Jetson Nano

3. Run Live: Load model, run inference loop, map x_target → steering

In [20]:
import torch
import numpy as np


# Placeholder Car
class Car:
    def __init__(self):
        self.steering = 0.0
        self.throttle = 0.0

car = Car()


# Placeholder preprocess
def preprocess(frame):
    return torch.tensor(frame, dtype=torch.float32).unsqueeze(0)


# Placeholder model
class DummyModel(torch.nn.Module):
    def forward(self, x):
        # Returns [x_target, y_target]
        return torch.tensor([[0.35, 0.80]])

model = DummyModel()

# Placeholder frame
frame = [1.0, 2.0, 3.0]


# Inference
with torch.no_grad():
    xy = model(preprocess(frame)).squeeze()

    x_t = xy[0].item()  # ∈ [-1, 1]

    # Implicit P Controller (Kp = 1)
    car.steering = float(np.clip(x_t, -1, 1))
    car.throttle = 0.18

print("Steering:", car.steering)
print("Throttle:", car.throttle)

Steering: 0.3499999940395355
Throttle: 0.18


#### 6 – Replacing P with PD Controller

PD control adds a derivative term to the P controller, improving response to changes in error and reducing overshoot. We will upgrade the Jet Racer steering from simple proportional to PD control to reduce overshoot and oscillation.

In [24]:
'''
The D term adds a "damping" effect to the steering, 
reducing overshoot and oscillation by considering the rate of change of error.
- Kd·de dampens the steering when error is decreasing fast, prevents over-correction.
• Typical Kd range: 0.1-0.2 at low speed
• Higher throttle → lower Kp or higher Kd

'''


def inference_loop(frame, kp, kd, prev_error):
    e = x_t # error vs. center (0.0)
    de = e - prev_error
    steer = Kp*e + Kd*de
    steer = float(np.clip(steer, -1.0, 1.0))
    car.steering = steer
    car.throttle = 0.18
    prev_error = e
    return e, de, steer


Kp = 0.8; Kd = 0.15
prev_error = 0.0
# In inference loop:
e, de, steer = inference_loop(frame, Kp, Kd, prev_error)
print(f"e={e:.3f}, de={de:.3f}, steer={steer:.3f}")

e=0.350, de=0.350, steer=0.332


In [ ]:
# Gain Tuning Workflow

'''
Set Kd = 0
Tune Kp until the car tracks but
still oscillates slightly
'''

Kd = 0.0
# Tune Kp until the car tracks but still oscillates slightly
kp_values = [0.5, 0.8, 1.0]
for Kp in kp_values:
    print(f"Testing Kp={Kp} with Kd={Kd}")
    # Run inference loop and observe behavior
    # Adjust Kp based on whether the car is understeering or oscillating too much
    e, de, steer = inference_loop(frame, Kp, Kd, prev_error)
    print(f"+ e={e:.3f}, de={de:.3f}, steer={steer:.3f}")

'''
Kp too high
Zigzag / strong oscillation
'''
    

Testing Kp=0.5 with Kd=0.0
+ e=0.350, de=0.350, steer=0.175
Testing Kp=0.8 with Kd=0.0
+ e=0.350, de=0.350, steer=0.280
Testing Kp=1.0 with Kd=0.0
+ e=0.350, de=0.350, steer=0.350


In [ ]:
''' Increase Kd
Raise gradually to reduce
oscillation without losing tracking'''
Kd_values = [0.05, 0.1, 0.15]
for Kd in Kd_values:
    print(f"Testing Kd={Kd} with Kp={Kp}")
    # Run inference loop and observe behavior
    # Adjust Kd based on whether the car is still oscillating or if it becomes too sluggish
    e, de, steer = inference_loop(frame, Kp, Kd, prev_error)
    print(f"+ e={e:.3f}, de={de:.3f}, steer={steer:.3f}")

'''
Kp too low
Slow tracking, exits line on curves
'''

Testing Kd=0.05 with Kp=1.0
+ e=0.350, de=0.350, steer=0.367
Testing Kd=0.1 with Kp=1.0
+ e=0.350, de=0.350, steer=0.385
Testing Kd=0.15 with Kp=1.0
+ e=0.350, de=0.350, steer=0.402


In [ ]:
'''Increase Throttle
Raise speed slightly, then re-
check and re-tune gains'''
throttle_values = [0.2, 0.25, 0.3]
for throttle in throttle_values:
    print(f"Testing throttle={throttle} with Kp={Kp}, Kd={Kd}")
    car.throttle = throttle
    # Run inference loop and observe behavior
    # Adjust gains based on whether the car can still track well at higher speed
    e, de, steer = inference_loop(frame, Kp, Kd, prev_error)
    print(f"+ e={e:.3f}, de={de:.3f}, steer={steer:.3f}")

'''
Kd too high
Jerky / sudden steering jolts
'''

Testing throttle=0.2 with Kp=1.0, Kd=0.15
+ e=0.350, de=0.350, steer=0.402
Testing throttle=0.25 with Kp=1.0, Kd=0.15
+ e=0.350, de=0.350, steer=0.402
Testing throttle=0.3 with Kp=1.0, Kd=0.15
+ e=0.350, de=0.350, steer=0.402


#### 7 – Turtlebot Line Following

Turtlebot is a popular ROS-based robot platform. We will implement line following using ROS topics to read camera images, compute error, and publish velocity commands to the robot.

Uses the iRobot Education Python SDK (or compatible driver). No ROS required — commands sent directly via Bluetooth/serial.

- Vision: Same `line_error_from_frame()` pipeline as Jet Racer
- PD Control
  - `Kp = 1.2, Kd = 0.1` → compute angular velocity ω pipeline is the same as Jet Racer, but we will publish `Twist` messages to the `/cmd_vel` topic instead of setting steering/throttle directly.

- `diff_drive_ik()`
    - Converts `(v, ω) → (v_L, v_R)` wheel speeds in mm/s

In [31]:
class Robot:
    def __init__(self):
        self.left_speed = 0
        self.right_speed = 0

    def set_wheel_speeds(self, left_speed, right_speed):
        self.left_speed = left_speed
        self.right_speed = right_speed
        print(f"Setting wheel speeds: Left={left_speed} mm/s, Right={right_speed} mm/s")

def diff_drive_ik(v, w, W=0.235):
    """W = Turtlebot 3 track width (m)"""
    v_R = v + w * W/2
    v_L = v - w * W/2
    return v_L, v_R

robot = Robot()
# If line lost → stop:
if e is None:
    robot.set_wheel_speeds(0, 0)

# Send command (m/s → mm/s):
v_L, v_R = diff_drive_ik(v_const, w)
robot.set_wheel_speeds(int(v_L*1000), int(v_R*1000))

'''
  Safety: finally Block
    Always wrap the main loop in try/finallyto guarantee the
    robot stops when the script exits.

    Skipping the finally block risks the robot continuing
    to move after the program crashes.

'''

Setting wheel speeds: Left=482 mm/s, Right=517 mm/s


'\n  Safety: finally Block\n    Always wrap the main loop in try/finallyto guarantee the\n    robot stops when the script exits.\n\n    Skipping the finally block risks the robot continuing\n    to move after the program crashes.\n\n'

#### Extension – Adaptive Throttle

Adaptive throttle reduces speed when the error is large, improving stability on tight curves. 

- When `|e| → 1` (large deviation), throttle drops toward min_throttle. 

- When `|e| → 0` (on track), full speed is restored.

In [33]:
'''
When |e| → 1 (large deviation), throttle drops toward
min_throttle. When |e| → 0 (on track), full speed is
restored.

Simple but highly effective in practice —
especially on tight curves.

'''

def adaptive_throttle(e, base_throttle=0.20, min_throttle=0.10):
    """
    Adaptive throttle based on deviation from the line.
    
    Args:
        e (float): Error signal representing deviation from the line.
        base_throttle (float): Base throttle value.
        min_throttle (float): Minimum throttle value.
    
    Returns:
        float: Adjusted throttle value.
    """
    # When |e| → 1 (large deviation), throttle drops toward
    # min_throttle. When |e| → 0 (on track), full speed is
    # restored.
    t = base_throttle * (1.0 - abs(e))
    return max(t, min_throttle)


# In loop:
throttle = adaptive_throttle(e)
car.throttle = throttle
print(f"Adaptive throttle: {throttle:.3f}")

Adaptive throttle: 0.130


Recap: We implemented line following using computer vision and control algorithms, starting with offline image processing and progressing to real-time control on the Jet Racer and Turtlebot platforms. The adaptive throttle extension further enhances performance on challenging tracks.